# Automated E-Commerce Review Tag Generator - DistilBERT Notebook

This notebook trains and evaluates DistilBERT models for aspect extraction and aspect-conditioned sentiment classification.
Dataset sources are referenced by links below; download the source files into the configured input directory before running the data-loading cells.

It is written to run on Kaggle with multiple GPUs, and it does not execute training here.

In [1]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import html
import json
import os
import random
import re
import sys
from typing import Any
from xml.etree import ElementTree as ET

import numpy as np

PROJECT_ROOT = Path('/kaggle/working/review_tag_generator') if Path('/kaggle/working').exists() else Path.cwd() / 'review_tag_generator'
SRC_ROOT = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_ROOT))

from enum import Enum
from pydantic import BaseModel, ConfigDict, Field

BIO_LABEL_TO_ID = {'O': 0, 'B-ASP': 1, 'I-ASP': 2}
BIO_ID_TO_LABEL = {value: key for key, value in BIO_LABEL_TO_ID.items()}
SENTIMENT_LABEL_TO_ID = {'negative': 0, 'neutral': 1, 'positive': 2}
SENTIMENT_ID_TO_LABEL = {value: key for key, value in SENTIMENT_LABEL_TO_ID.items()}

class SentimentLabel(str, Enum):
    negative = 'negative'
    neutral = 'neutral'
    positive = 'positive'

class AspectAnnotation(BaseModel):
    model_config = ConfigDict(extra='ignore')
    aspect: str
    normalized_aspect: str
    opinion: str = ''
    sentiment: SentimentLabel
    start_char: int
    end_char: int

class Review(BaseModel):
    model_config = ConfigDict(extra='ignore')
    review_id: str
    product_id: str
    product_category: str
    review_text: str
    rating: float | int | None = 0
    helpful_votes: int | None = 0
    timestamp: int | str | None = 0
    annotations: list[AspectAnnotation] = Field(default_factory=list)

class AspectPrediction(BaseModel):
    aspect: str
    start_char: int
    end_char: int
    confidence: float

class SentimentPrediction(BaseModel):
    sentiment: SentimentLabel
    confidence: float

try:
    import torch
    from datasets import Dataset
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
    )
    from sklearn.pipeline import Pipeline
    from transformers import (
        AutoConfig,
        AutoModelForSequenceClassification,
        AutoModelForTokenClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        DataCollatorForTokenClassification,
        EarlyStoppingCallback,
        Trainer,
        TrainingArguments,
    )
    TRANSFORMER_AVAILABLE = True
    TRANSFORMER_IMPORT_ERROR = None
except ImportError as exc:
    TRANSFORMER_AVAILABLE = False
    TRANSFORMER_IMPORT_ERROR = str(exc)

MODEL_NAME = 'distilbert-base-uncased'
SEED = 42
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
NUM_EPOCHS = 4
ASPECT_NUM_EPOCHS = 8
ASPECT_LEARNING_RATE = 3e-5
ASPECT_CLASS_WEIGHTS = [0.25, 1.5, 1.5]
ASPECT_RUN_NAME = 'distilbert_weighted_lr3e5'
WARMUP_STEPS = 0
ALLOW_SMOKE_FALLBACK = False
DATASET_LINKS = {
    'laptop_acos': 'https://github.com/NUSTM/ACOS/tree/main/data/Laptop-ACOS',
    'semeval_2014_laptop': 'https://aclanthology.org/S14-2004/',
}

DATA_DIR = PROJECT_ROOT / 'data'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed' / 'distilbert'
MODELS_DIR = PROJECT_ROOT / 'models'
EVIDENCE_DIR = PROJECT_ROOT / 'audit' / 'evidence'
DATASET_ROOTS = (
    Path('/kaggle/input/datasets/cyrilsabu/laptop-train'),
    Path('/kaggle/input'),
    PROJECT_ROOT,
)


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if TRANSFORMER_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def supports_tf32() -> bool:
    return bool(torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_directory(path: Path) -> str:
    digest = hashlib.sha256()
    for file_path in sorted(p for p in path.rglob('*') if p.is_file()):
        digest.update(str(file_path.relative_to(path)).encode('utf-8'))
        digest.update(file_path.read_bytes())
    return digest.hexdigest()


def write_json(path: Path, payload: dict) -> dict:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return payload


def write_jsonl(path: Path, rows: list[dict]) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text('\n'.join(json.dumps(row, sort_keys=True) for row in rows) + ('\n' if rows else ''), encoding='utf-8')
    return path


def model_ready() -> bool:
    required = [
        INTERIM_DIR / 'laptop_acos' / 'laptop_acos_common.jsonl',
        INTERIM_DIR / 'semeval_2014_laptop' / 'train.jsonl',
        INTERIM_DIR / 'semeval_2014_laptop' / 'test.jsonl',
    ]
    return all(path.exists() for path in required)


set_seed(SEED)


## Data Loading

This section reads the converter outputs, preserves official source splits, and builds a deduplicated processed dataset with train, validation, and test partitions.

In [2]:
def discover_dataset_file(filename_or_names, roots: tuple[Path, ...] = DATASET_ROOTS) -> Path:
    names = (filename_or_names,) if isinstance(filename_or_names, str) else tuple(filename_or_names)
    matches: list[Path] = []
    for root in roots:
        if not root.exists():
            continue
        for name in names:
            matches.extend(path for path in root.rglob(name) if path.is_file())
    if not matches:
        raise FileNotFoundError('Missing official dataset file(s): ' + ', '.join(names))
    return sorted(matches)[0]


def clean_review_text(text: str) -> str:
    cleaned = html.unescape(text or '')
    cleaned = re.sub(r'<[^>]+>', ' ', cleaned)
    cleaned = re.sub(r'https?://\S+|www\.\S+', ' ', cleaned)
    cleaned = re.sub(r'\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b', ' ', cleaned)
    cleaned = ''.join(ch for ch in cleaned if ch.isprintable() or ch in '\n\t')
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


def normalize_sentiment_label(raw: str | None) -> str | None:
    if raw is None:
        return None
    value = str(raw).strip().lower()
    mapping = {
        '0': 'negative',
        'negative': 'negative',
        'neg': 'negative',
        '1': 'neutral',
        'neutral': 'neutral',
        'neu': 'neutral',
        '2': 'positive',
        'positive': 'positive',
        'pos': 'positive',
    }
    return mapping.get(value)


def split_raw_record(line: str) -> tuple[str, list[str]]:
    raw = line.rstrip('\n')
    if '\t' in raw:
        parts = [part.strip() for part in raw.split('\t') if part.strip()]
        if len(parts) >= 2:
            return parts[0], parts[1:]
    if '####' in raw:
        parts = [part.strip() for part in raw.split('####') if part.strip()]
        if len(parts) >= 2:
            return parts[0], parts[1:]

    # Official Laptop-ACOS rows append whitespace-separated quads:
    # aspect_span category sentiment opinion_span.
    indexed_quad = re.compile(r'-?\d+,-?\d+\s+\S+\s+[012]\s+-?\d+,-?\d+')
    match = indexed_quad.search(raw)
    if match:
        review_text = raw[:match.start()].strip()
        annotation_text = raw[match.start():]
        annotations = [quad.group(0) for quad in indexed_quad.finditer(annotation_text)]
        if review_text and annotations:
            return review_text, annotations

    raise ValueError(f'Cannot split raw ACOS record: {raw[:120]!r}')


def indexed_span_text(text: str, span: str) -> str | None:
    if span == '-1,-1':
        return None
    try:
        start, end = (int(value) for value in span.split(',', maxsplit=1))
    except (TypeError, ValueError):
        return None
    tokens = list(re.finditer(r'\S+', text))
    if start < 0 or end <= start or end > len(tokens):
        return None
    return text[tokens[start].start():tokens[end - 1].end()]


def parse_indexed_acos_annotation(token: str, review_text: str) -> tuple[str, str, str] | None:
    parts = token.split()
    if len(parts) != 4 or not re.fullmatch(r'-?\d+,-?\d+', parts[0]) or not re.fullmatch(r'-?\d+,-?\d+', parts[3]):
        return None
    sentiment = normalize_sentiment_label(parts[2])
    aspect = indexed_span_text(review_text, parts[0])
    opinion = indexed_span_text(review_text, parts[3]) or ''
    if sentiment is None or aspect is None:
        return None
    return clean_review_text(aspect), clean_review_text(opinion), sentiment


def parse_acos_annotation(token: str) -> tuple[str, str, str] | None:
    raw = token.strip().strip('[](){}')
    if not raw:
        return None
    parts = None
    for sep in ('####', '|||', '###', '\t', '|', ';'):
        if sep in raw:
            split_parts = [part.strip() for part in raw.split(sep) if part.strip()]
            if len(split_parts) == 4:
                parts = split_parts
                break
    if parts is None:
        split_parts = [part.strip() for part in raw.split() if part.strip()]
        if len(split_parts) == 4:
            parts = split_parts
    if parts is None:
        return None

    sentiment_idx = next((idx for idx, part in enumerate(parts) if normalize_sentiment_label(part)), None)
    if sentiment_idx is None:
        return None
    sentiment = normalize_sentiment_label(parts[sentiment_idx])
    if sentiment is None:
        return None

    if sentiment_idx == 2:
        aspect = parts[0]
        opinion = parts[3]
    elif sentiment_idx == 3:
        aspect = parts[0]
        opinion = parts[2]
    else:
        return None

    aspect = clean_review_text(aspect)
    opinion = clean_review_text(opinion)
    if not aspect or aspect.lower() in {'null', 'none', 'nan'}:
        return None
    return aspect, opinion, sentiment


def locate_span(text: str, target: str, occupied: set[tuple[int, int]] | None = None) -> tuple[int, int] | None:
    occupied = occupied or set()
    pattern = re.compile(re.escape(target), flags=re.IGNORECASE)
    for match in pattern.finditer(text):
        span = (match.start(), match.end())
        if span not in occupied:
            return span
    return None


def build_review_payload(*, review_id: str, product_id: str, review_text: str, product_category: str, annotations: list[dict[str, Any]], rating: float | int | None = 0, helpful_votes: int | None = 0, timestamp: int | str | None = 0) -> Review:
    payload = {
        'review_id': review_id,
        'product_id': product_id,
        'product_category': product_category,
        'review_text': review_text,
        'rating': rating,
        'helpful_votes': helpful_votes,
        'timestamp': timestamp,
        'annotations': annotations,
    }
    return Review.model_validate(payload)


def load_laptop_acos_reviews(path: Path, source_split: str) -> list[Review]:
    reviews: list[Review] = []
    for line_no, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
        if not line.strip():
            continue
        review_text, annotation_tokens = split_raw_record(line)
        review_text = clean_review_text(review_text)
        used_spans: set[tuple[int, int]] = set()
        annotations: list[dict[str, Any]] = []
        for token in annotation_tokens:
            parsed = parse_indexed_acos_annotation(token, review_text) or parse_acos_annotation(token)
            if parsed is None:
                continue
            aspect_text, opinion_text, sentiment = parsed
            span = locate_span(review_text, aspect_text, used_spans)
            if span is None:
                continue
            used_spans.add(span)
            start_char, end_char = span
            annotations.append({
                'aspect': review_text[start_char:end_char],
                'normalized_aspect': aspect_text.casefold(),
                'opinion': opinion_text,
                'sentiment': sentiment,
                'start_char': start_char,
                'end_char': end_char,
            })
        reviews.append(build_review_payload(
            review_id=f'laptop_acos:{source_split}:{line_no:05d}',
            product_id=f'laptop_acos:{source_split}:{line_no:05d}',
            review_text=review_text,
            product_category='laptop',
            annotations=annotations,
            rating=0,
            helpful_votes=0,
            timestamp=0,
        ))
    return reviews


def load_semeval_laptop_reviews(path: Path, source_split: str) -> list[Review]:
    reviews: list[Review] = []
    tree = ET.parse(path)
    root = tree.getroot()
    for sentence in root.iter('sentence'):
        sentence_id = sentence.attrib.get('id', str(len(reviews) + 1))
        text_node = sentence.find('text')
        review_text = clean_review_text(text_node.text if text_node is not None and text_node.text is not None else '')
        annotations: list[dict[str, Any]] = []
        used_spans: set[tuple[int, int]] = set()
        aspect_terms = sentence.find('aspectTerms')
        if aspect_terms is not None:
            for aspect_term in aspect_terms.iter('aspectTerm'):
                term = clean_review_text(aspect_term.attrib.get('term', ''))
                sentiment = normalize_sentiment_label(aspect_term.attrib.get('polarity'))
                if sentiment is None or not term or term.lower() in {'null', 'none', 'nan'}:
                    continue
                start_attr = aspect_term.attrib.get('from')
                end_attr = aspect_term.attrib.get('to')
                span = None
                if start_attr is not None and end_attr is not None:
                    try:
                        start_char = int(start_attr)
                        end_char = int(end_attr)
                        if 0 <= start_char < end_char <= len(review_text) and review_text[start_char:end_char].strip():
                            span = (start_char, end_char)
                    except ValueError:
                        span = None
                if span is None:
                    span = locate_span(review_text, term, used_spans)
                if span is None:
                    continue
                used_spans.add(span)
                start_char, end_char = span
                annotations.append({
                    'aspect': review_text[start_char:end_char],
                    'normalized_aspect': term.casefold(),
                    'opinion': '',
                    'sentiment': sentiment,
                    'start_char': start_char,
                    'end_char': end_char,
                })
        reviews.append(build_review_payload(
            review_id=f'semeval_2014_laptop:{source_split}:{sentence_id}',
            product_id=f'semeval_2014_laptop:{source_split}:{sentence_id}',
            review_text=review_text,
            product_category='laptop',
            annotations=annotations,
            rating=0,
            helpful_votes=0,
            timestamp=0,
        ))
    return reviews


def load_common_reviews() -> list[Review]:
    sources = [
        (load_laptop_acos_reviews, 'laptop_quad_train.tsv', 'train'),
        (load_laptop_acos_reviews, 'laptop_quad_dev.tsv', 'dev'),
        (load_laptop_acos_reviews, 'laptop_quad_test.tsv', 'test'),
        (load_semeval_laptop_reviews, ('Laptop_Train.xml', 'Laptop_Train_v2.xml'), 'train'),
        (load_semeval_laptop_reviews, ('Laptops_Test_Gold.xml', 'Laptops_Test_Gold_v2.xml'), 'test'),
    ]

    reviews: list[Review] = []
    for loader, filename_or_names, source_split in sources:
        path = discover_dataset_file(filename_or_names)
        reviews.extend(loader(path, source_split))
    return reviews


def split_from_review_id(review_id: str) -> str:
    if ':train:' in review_id:
        return 'train'
    if ':dev:' in review_id or ':validation:' in review_id:
        return 'validation'
    if ':test:' in review_id:
        return 'test'
    return 'train'


def split_priority(review_id: str) -> int:
    split = split_from_review_id(review_id)
    return {'test': 0, 'validation': 1, 'train': 2}.get(split, 2)


def normalized_text_key(text: str) -> str:
    return sha256_text(' '.join(text.casefold().split()))


def build_processed_dataset(reviews: list[Review], output_dir: Path) -> dict:
    grouped: dict[str, Review] = {}
    rejected = []
    split_rows = {'train': [], 'validation': [], 'test': []}
    duplicate_manifest = []
    source_counts = Counter()
    sentiment_counts = Counter()
    aspect_counts = Counter()

    ordered_reviews = sorted(reviews, key=lambda r: (split_priority(r.review_id), r.review_id))
    for review in ordered_reviews:
        cleaned_text = clean_review_text(review.review_text)
        if not cleaned_text:
            rejected.append({'review_id': review.review_id, 'reason': 'empty_cleaned_text'})
            continue

        dedup_key = normalized_text_key(cleaned_text)
        if dedup_key in grouped:
            duplicate_manifest.append({
                'dedup_key': dedup_key,
                'kept_review_id': grouped[dedup_key].review_id,
                'dropped_review_id': review.review_id,
                'reason': 'exact_duplicate_after_normalization',
            })
            continue

        processed = review.model_copy(update={'review_text': cleaned_text})
        grouped[dedup_key] = processed
        split = split_from_review_id(review.review_id)
        split_rows[split].append(processed)
        source_counts[split] += 1
        for ann in processed.annotations:
            sentiment_counts[getattr(ann.sentiment, 'value', str(ann.sentiment)).lower()] += 1
            aspect_counts[str(ann.aspect).casefold()] += 1

    if not split_rows['validation'] and split_rows['train']:
        train_rows = split_rows['train']
        validation_rows = []
        retained_train = []
        for review in train_rows:
            bucket = int(sha256_text(review.review_id)[:8], 16) % 10
            if bucket == 0:
                validation_rows.append(review)
            else:
                retained_train.append(review)
        if not validation_rows and retained_train:
            validation_rows.append(retained_train.pop(0))
        split_rows['train'] = retained_train
        split_rows['validation'] = validation_rows

    for split in split_rows:
        split_rows[split] = sorted(split_rows[split], key=lambda r: r.review_id)

    output_dir.mkdir(parents=True, exist_ok=True)
    for split, rows in split_rows.items():
        write_jsonl(output_dir / f'{split}.jsonl', [r.model_dump(mode='json') for r in rows])

    write_jsonl(output_dir / 'rejected.jsonl', rejected)
    write_jsonl(output_dir / 'dedup_manifest.jsonl', duplicate_manifest)

    statistics = {
        'input_reviews': len(reviews),
        'accepted_reviews': sum(len(rows) for rows in split_rows.values()),
        'rejected_reviews': len(rejected),
        'duplicate_count': len(duplicate_manifest),
        'split_counts': {split: len(rows) for split, rows in split_rows.items()},
        'source_counts': dict(source_counts),
        'sentiment_counts': dict(sentiment_counts),
        'aspect_counts': dict(aspect_counts),
        'dataset_mode': 'linked_sources',
    }
    write_json(output_dir / 'statistics.json', statistics)
    split_hashes = {split: sha256_file(output_dir / f'{split}.jsonl') for split in split_rows}
    write_json(output_dir / 'split_manifest.json', {
        'manifest_version': '1.0',
        'seed': SEED,
        'dataset_mode': 'linked_sources',
        'split_hashes': split_hashes,
        'statistics': statistics,
    })
    return {'splits': split_rows, 'statistics': statistics, 'split_hashes': split_hashes}


def model_ready() -> bool:
    required = [
        ('laptop_quad_train.tsv',),
        ('laptop_quad_dev.tsv',),
        ('laptop_quad_test.tsv',),
        ('Laptop_Train.xml', 'Laptop_Train_v2.xml'),
        ('Laptops_Test_Gold.xml', 'Laptops_Test_Gold_v2.xml'),
    ]
    try:
        for filenames in required:
            discover_dataset_file(filenames)
    except FileNotFoundError:
        return False
    return True


all_reviews = load_common_reviews()
processed = build_processed_dataset(all_reviews, PROCESSED_DIR)
processed_splits = processed['splits']
processed_stats = processed['statistics']


## Classical Baselines

These baselines give a real comparison point for the transformer models: a rule-based aspect extractor and a TF-IDF plus logistic-regression sentiment classifier conditioned on the requested aspect.

In [3]:
def build_aspect_sentiment_examples(reviews: list[Review]) -> list[dict]:
    rows = []
    for review in reviews:
        for ann in review.annotations:
            rows.append({
                'review_id': review.review_id,
                'product_id': review.product_id,
                'review_text': review.review_text,
                'aspect_text': ann.aspect,
                'start_char': ann.start_char,
                'end_char': ann.end_char,
                'label': ann.sentiment.value,
                'label_id': SENTIMENT_LABEL_TO_ID[ann.sentiment.value],
            })
    return rows


def format_aspect_pair(review_text: str, aspect_text: str) -> str:
    return f'aspect: {aspect_text} [SEP] review: {review_text}'


def train_logistic_sentiment_baseline(train_examples: list[dict]) -> Pipeline:
    X = [format_aspect_pair(row['review_text'], row['aspect_text']) for row in train_examples]
    y = [row['label'] for row in train_examples]
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1, 2), sublinear_tf=True, max_features=50000)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced', multi_class='auto')),
    ])
    pipeline.fit(X, y)
    return pipeline


def evaluate_sentiment_baseline(model, examples: list[dict], split_name: str) -> dict:
    X = [format_aspect_pair(row['review_text'], row['aspect_text']) for row in examples]
    y_true = [row['label'] for row in examples]
    y_pred = list(model.predict(X))
    labels = ['positive', 'neutral', 'negative']
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    per_class = {}
    for label in labels:
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[label], average='macro', zero_division=0)
        per_class[label] = {'precision': float(p), 'recall': float(r), 'f1': float(f)}
    return {
        'split': split_name,
        'n_examples': len(examples),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'per_class': per_class,
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def rule_extract_aspects(text: str, aspect_terms: list[str]) -> list[AspectPrediction]:
    predictions = []
    seen = set()
    for term in sorted(set(aspect_terms), key=len, reverse=True):
        pattern = re.compile(r'(?<!\w)' + re.escape(term) + r'(?!\w)', flags=re.IGNORECASE)
        for match in pattern.finditer(text):
            span = (match.start(), match.end())
            if span in seen:
                continue
            seen.add(span)
            predictions.append(AspectPrediction(aspect=text[span[0]:span[1]], start_char=span[0], end_char=span[1], confidence=0.65))
    return sorted(predictions, key=lambda p: (p.start_char, p.end_char))


def evaluate_aspect_rule_baseline(reviews: list[Review], aspect_terms: list[str], split_name: str) -> dict:
    tp = fp = fn = 0
    for review in reviews:
        gold = {(ann.start_char, ann.end_char) for ann in review.annotations}
        pred = {(p.start_char, p.end_char) for p in rule_extract_aspects(review.review_text, aspect_terms)}
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        'split': split_name,
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'tp': tp,
        'fp': fp,
        'fn': fn,
    }


train_reviews = processed_splits['train']
validation_reviews = processed_splits['validation']
test_reviews = processed_splits['test']

train_examples = build_aspect_sentiment_examples(train_reviews)
validation_examples = build_aspect_sentiment_examples(validation_reviews)
test_examples = build_aspect_sentiment_examples(test_reviews)

rule_aspect_terms = [ann.aspect.lower() for review in train_reviews for ann in review.annotations]
rule_aspect_metrics = {
    'validation': evaluate_aspect_rule_baseline(validation_reviews, rule_aspect_terms, 'validation'),
    'test': evaluate_aspect_rule_baseline(test_reviews, rule_aspect_terms, 'test'),
}

sentiment_baseline = train_logistic_sentiment_baseline(train_examples)
sentiment_baseline_metrics = {
    'validation': evaluate_sentiment_baseline(sentiment_baseline, validation_examples, 'validation'),
    'test': evaluate_sentiment_baseline(sentiment_baseline, test_examples, 'test'),
}

baseline_dir = PROJECT_ROOT / 'models' / 'baselines'
baseline_dir.mkdir(parents=True, exist_ok=True)
write_json(baseline_dir / 'aspect_rule_metrics.json', rule_aspect_metrics)
write_json(baseline_dir / 'tfidf_logistic_metrics.json', sentiment_baseline_metrics)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


{'validation': {'split': 'validation',
  'n_examples': 309,
  'accuracy': 0.7605177993527508,
  'macro_precision': 0.601660335653485,
  'macro_recall': 0.5963374597150688,
  'macro_f1': 0.598781937017231,
  'per_class': {'positive': {'precision': 0.8457446808510638,
    'recall': 0.8548387096774194,
    'f1': 0.8502673796791443},
   'neutral': {'precision': 0.2631578947368421,
    'recall': 0.23809523809523808,
    'f1': 0.25},
   'negative': {'precision': 0.696078431372549,
    'recall': 0.696078431372549,
    'f1': 0.696078431372549}},
  'confusion_matrix': [[159, 7, 20], [5, 5, 11], [24, 7, 71]]},
 'test': {'split': 'test',
  'n_examples': 1445,
  'accuracy': 0.7314878892733564,
  'macro_precision': 0.6339189209427258,
  'macro_recall': 0.6213400957277391,
  'macro_f1': 0.6173789003975411,
  'per_class': {'positive': {'precision': 0.8364312267657993,
    'recall': 0.8323057953144266,
    'f1': 0.8343634116192831},
   'neutral': {'precision': 0.424,
    'recall': 0.24651162790697675,

## Aspect Alignment

This section prepares DistilBERT token-classification inputs and computes BIO labels from character spans with the fast tokenizer offset mapping.

In [4]:
def load_aspect_tokenizer(model_name: str = MODEL_NAME):
    if not TRANSFORMER_AVAILABLE:
        raise RuntimeError(TRANSFORMER_IMPORT_ERROR or 'transformers unavailable')
    return AutoTokenizer.from_pretrained(model_name, use_fast=True)


def align_bio_labels_hf(review: Review, tokenizer, max_length: int = MAX_LENGTH) -> dict:
    encoding = tokenizer(review.review_text, return_offsets_mapping=True, truncation=True, max_length=max_length)
    offsets = encoding['offset_mapping']
    labels = [-100 if start == 0 and end == 0 else 0 for start, end in offsets]
    for annotation in sorted(review.annotations, key=lambda ann: (ann.start_char, ann.end_char)):
        covered = [idx for idx, (start, end) in enumerate(offsets) if not (start == 0 and end == 0) and start < annotation.end_char and end > annotation.start_char]
        if not covered:
            raise ValueError(f'truncated annotation for {review.review_id}: {annotation.aspect}')
        labels[covered[0]] = BIO_LABEL_TO_ID['B-ASP']
        for idx in covered[1:]:
            labels[idx] = BIO_LABEL_TO_ID['I-ASP']
    return {
        'review_id': review.review_id,
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels': labels,
        'offset_mapping': offsets,
        'text': review.review_text,
    }


def build_hf_aspect_dataset(reviews: list[Review], tokenizer, max_length: int = MAX_LENGTH):
    rows = [align_bio_labels_hf(review, tokenizer, max_length=max_length) for review in reviews]
    dataset_rows = [{'input_ids': row['input_ids'], 'attention_mask': row['attention_mask'], 'labels': row['labels']} for row in rows]
    return Dataset.from_list(dataset_rows), rows


def bio_label_sequences_from_predictions(predictions: np.ndarray, labels: np.ndarray) -> tuple[list[list[str]], list[list[str]]]:
    predicted_sequences: list[list[str]] = []
    gold_sequences: list[list[str]] = []
    for pred_row, label_row in zip(predictions, labels):
        pred_labels = []
        gold_labels = []
        for pred_id, gold_id in zip(pred_row, label_row):
            if gold_id == -100:
                continue
            pred_labels.append(BIO_ID_TO_LABEL[int(pred_id)])
            gold_labels.append(BIO_ID_TO_LABEL[int(gold_id)])
        predicted_sequences.append(pred_labels)
        gold_sequences.append(gold_labels)
    return predicted_sequences, gold_sequences


def spans_from_bio_labels(labels: list[str]) -> set[tuple[int, int]]:
    spans: set[tuple[int, int]] = set()
    start = None
    for idx, label in enumerate(labels):
        if label == 'B-ASP':
            if start is not None:
                spans.add((start, idx))
            start = idx
        elif label == 'I-ASP':
            if start is None:
                start = idx
        else:
            if start is not None:
                spans.add((start, idx))
                start = None
    if start is not None:
        spans.add((start, len(labels)))
    return spans


def compute_exact_span_metrics(predicted_sequences: list[list[str]], gold_sequences: list[list[str]]) -> dict[str, float]:
    tp = fp = fn = 0
    for pred_labels, gold_labels in zip(predicted_sequences, gold_sequences):
        pred_spans = spans_from_bio_labels(pred_labels)
        gold_spans = spans_from_bio_labels(gold_labels)
        tp += len(pred_spans & gold_spans)
        fp += len(pred_spans - gold_spans)
        fn += len(gold_spans - pred_spans)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'precision': float(precision), 'recall': float(recall), 'f1': float(f1), 'tp': tp, 'fp': fp, 'fn': fn}


def compute_span_metrics(eval_prediction) -> dict[str, float]:
    predictions, labels = eval_prediction
    predictions = np.argmax(predictions, axis=-1)
    predicted_sequences, gold_sequences = bio_label_sequences_from_predictions(predictions, labels)
    token_true = [label for seq in gold_sequences for label in seq]
    token_pred = [label for seq in predicted_sequences for label in seq]
    token_precision, token_recall, token_f1, _ = precision_recall_fscore_support(token_true, token_pred, labels=['B-ASP', 'I-ASP', 'O'], average='macro', zero_division=0)
    token_accuracy = accuracy_score(token_true, token_pred)
    exact_span = compute_exact_span_metrics(predicted_sequences, gold_sequences)
    return {
        'token_precision': float(token_precision),
        'token_recall': float(token_recall),
        'token_f1': float(token_f1),
        'token_accuracy': float(token_accuracy),
        'span_precision': exact_span['precision'],
        'span_recall': exact_span['recall'],
        'span_f1': exact_span['f1'],
        'span_accuracy': float(token_accuracy),
    }


def create_aspect_model(model_name: str = MODEL_NAME):
    config = AutoConfig.from_pretrained(model_name, num_labels=len(BIO_LABEL_TO_ID), id2label=BIO_ID_TO_LABEL, label2id=BIO_LABEL_TO_ID)
    return AutoModelForTokenClassification.from_pretrained(model_name, config=config)


def make_aspect_training_args(output_dir: Path, *, epochs: float = ASPECT_NUM_EPOCHS, learning_rate: float = ASPECT_LEARNING_RATE, train_batch_size: int = TRAIN_BATCH_SIZE, eval_batch_size: int = EVAL_BATCH_SIZE, gradient_accumulation_steps: int = 1, seed: int = SEED):
    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        weight_decay=0.01,
        warmup_steps=WARMUP_STEPS,
        optim='adamw_torch',
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='span_f1',
        greater_is_better=True,
        save_total_limit=2,
        fp16=bool(torch.cuda.is_available()),
        bf16=False,
        tf32=supports_tf32(),
        dataloader_num_workers=2,
        ddp_find_unused_parameters=False,
        seed=seed,
        data_seed=seed,
        report_to=[],
        remove_unused_columns=False,
    )


class WeightedTokenClassificationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        weights = torch.tensor(ASPECT_CLASS_WEIGHTS, dtype=torch.float, device=outputs.logits.device)
        loss = torch.nn.CrossEntropyLoss(weight=weights, ignore_index=-100)(
            outputs.logits.view(-1, outputs.logits.size(-1)),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss


def build_aspect_training_artifacts():
    if not TRANSFORMER_AVAILABLE:
        raise RuntimeError(TRANSFORMER_IMPORT_ERROR or 'transformers unavailable')
    tokenizer = load_aspect_tokenizer()
    train_ds, train_rows = build_hf_aspect_dataset(train_reviews, tokenizer)
    validation_ds, validation_rows = build_hf_aspect_dataset(validation_reviews, tokenizer)
    test_ds, test_rows = build_hf_aspect_dataset(test_reviews, tokenizer)
    output_dir = MODELS_DIR / 'aspect_extractor' / ASPECT_RUN_NAME
    output_dir.mkdir(parents=True, exist_ok=True)
    model = create_aspect_model()
    args = make_aspect_training_args(output_dir)
    trainer = WeightedTokenClassificationTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=validation_ds,
        data_collator=DataCollatorForTokenClassification(tokenizer, label_pad_token_id=-100, pad_to_multiple_of=8 if torch.cuda.is_available() else None),
        compute_metrics=compute_span_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    return {
        'trainer': trainer,
        'tokenizer': tokenizer,
        'datasets': {'train': train_ds, 'validation': validation_ds, 'test': test_ds},
        'rows': {'train': train_rows, 'validation': validation_rows, 'test': test_rows},
        'output_dir': output_dir,
    }


aspect_artifacts = build_aspect_training_artifacts()


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Aspect Training

This section trains or loads the real DistilBERT token-classification checkpoint, then writes the A2 manifests and metrics artifacts.

In [5]:
def reconstruct_aspect_predictions(text: str, tokenizer, logits: np.ndarray, offsets: list[tuple[int, int]]) -> list[AspectPrediction]:
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    pred_ids = probs.argmax(axis=-1)
    spans = []
    current = None
    for idx, (start, end) in enumerate(offsets):
        if start == 0 and end == 0:
            continue
        label = BIO_ID_TO_LABEL[int(pred_ids[idx])]
        token_conf = float(probs[idx, pred_ids[idx]])
        if label == 'B-ASP':
            if current:
                spans.append(current)
            current = {'start': start, 'end': end, 'scores': [token_conf]}
        elif label == 'I-ASP' and current is not None:
            current['end'] = end
            current['scores'].append(token_conf)
        else:
            if current:
                spans.append(current)
                current = None
    if current:
        spans.append(current)
    predictions = []
    for span in spans:
        aspect_text = text[span['start']:span['end']]
        if aspect_text.strip():
            predictions.append(AspectPrediction(aspect=aspect_text, start_char=span['start'], end_char=span['end'], confidence=float(np.mean(span['scores']))))
    deduped = []
    seen = set()
    for pred in predictions:
        key = (pred.start_char, pred.end_char)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(pred)
    return deduped


def evaluate_aspect_predictions(trainer, reviews: list[Review], tokenizer, split_name: str) -> dict:
    dataset, rows = build_hf_aspect_dataset(reviews, tokenizer)
    prediction_output = trainer.predict(dataset)
    logits = prediction_output.predictions
    labels = prediction_output.label_ids
    metrics = compute_span_metrics((logits, labels))
    exact_tp = exact_fp = exact_fn = 0
    errors = []
    for review, row, logit_row in zip(reviews, rows, logits):
        preds = reconstruct_aspect_predictions(review.review_text, tokenizer, logit_row, row['offset_mapping'])
        gold = {(a.start_char, a.end_char, a.aspect.lower()) for a in review.annotations}
        pred = {(p.start_char, p.end_char, p.aspect.lower()) for p in preds}
        exact_tp += len(gold & pred)
        exact_fp += len(pred - gold)
        exact_fn += len(gold - pred)
        if gold != pred:
            errors.append({'review_id': review.review_id, 'gold': sorted(list(gold)), 'pred': sorted(list(pred))})
    precision = exact_tp / (exact_tp + exact_fp) if exact_tp + exact_fp else 0.0
    recall = exact_tp / (exact_tp + exact_fn) if exact_tp + exact_fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'split': split_name, 'span': metrics, 'exact_span': {'precision': precision, 'recall': recall, 'f1': f1}, 'errors': errors}


def train_aspect_transformer():
    artifacts = aspect_artifacts
    trainer = artifacts['trainer']
    train_result = trainer.train()
    trainer.save_model(str(artifacts['output_dir'] / 'best'))
    artifacts['tokenizer'].save_pretrained(str(artifacts['output_dir'] / 'best'))
    validation_metrics = trainer.evaluate(artifacts['datasets']['validation'])
    test_metrics = evaluate_aspect_predictions(trainer, test_reviews, artifacts['tokenizer'], 'test')
    manifest = {
        'manifest_version': '1.0',
        'task_id': 'A2_aspect_extraction',
        'mode': 'transformer',
        'run_id': f'aspect_distilbert_seed{SEED}',
        'reason': 'transformer_training_available',
        'model': {'name': MODEL_NAME, 'revision': None, 'tokenizer_name': MODEL_NAME, 'tokenizer_revision': None},
        'seed': SEED,
        'config': {'epochs': ASPECT_NUM_EPOCHS, 'learning_rate': ASPECT_LEARNING_RATE, 'train_batch_size': TRAIN_BATCH_SIZE, 'eval_batch_size': EVAL_BATCH_SIZE, 'max_length': MAX_LENGTH},
        'dataset_hashes': {
            'train': sha256_text(json.dumps([r.model_dump(mode='json') for r in train_reviews], sort_keys=True)),
            'validation': sha256_text(json.dumps([r.model_dump(mode='json') for r in validation_reviews], sort_keys=True)),
            'test': sha256_text(json.dumps([r.model_dump(mode='json') for r in test_reviews], sort_keys=True)),
        },
        'checkpoint': {'path': str(artifacts['output_dir'] / 'best'), 'sha256': sha256_directory(artifacts['output_dir'] / 'best')},
        'metrics_paths': {
            'validation': str(EVIDENCE_DIR / 'A2' / 'aspect_validation.json'),
            'test': str(EVIDENCE_DIR / 'A2' / 'aspect_test.json'),
        },
        'runtime': {'cuda_available': bool(torch.cuda.is_available()), 'gpu_count': int(torch.cuda.device_count())},
    }
    write_json(EVIDENCE_DIR / 'A2' / 'aspect_training_manifest.json', manifest)
    write_json(EVIDENCE_DIR / 'A2' / 'aspect_validation.json', validation_metrics)
    write_json(EVIDENCE_DIR / 'A2' / 'aspect_test.json', test_metrics)
    return {'manifest': manifest, 'validation_metrics': validation_metrics, 'test_metrics': test_metrics, 'trainer': trainer, 'tokenizer': artifacts['tokenizer']}


aspect_training_result = train_aspect_transformer()
a2_summary = {
    'exact_span_f1': aspect_training_result['test_metrics']['exact_span']['f1'],
    'span_f1': aspect_training_result['test_metrics']['span']['span_f1'],
    'baseline_exact_span_f1': rule_aspect_metrics['test']['f1'],
    'passed': aspect_training_result['test_metrics']['exact_span']['f1'] >= 0.70 and aspect_training_result['test_metrics']['exact_span']['f1'] > rule_aspect_metrics['test']['f1'],
}


Epoch,Training Loss,Validation Loss,Token Precision,Token Recall,Token F1,Token Accuracy,Span Precision,Span Recall,Span F1,Span Accuracy
1,0.332590,0.224569,0.710440,0.914662,0.786313,0.927950,0.516634,0.854369,0.643902,0.927950
2,0.181452,0.243937,0.726168,0.899205,0.794022,0.934618,0.534737,0.822006,0.647959,0.934618
3,0.126626,0.284222,0.768954,0.881750,0.817282,0.945360,0.577626,0.818770,0.677376,0.945360
4,0.090899,0.317690,0.777648,0.879948,0.822329,0.948509,0.612440,0.828479,0.704264,0.948509
5,0.068248,0.387529,0.789129,0.858666,0.820809,0.949250,0.625323,0.783172,0.695402,0.949250
6,0.052357,0.404752,0.804144,0.866772,0.832390,0.953325,0.637056,0.812298,0.714083,0.953325
7,0.041350,0.428121,0.802238,0.863262,0.829743,0.952028,0.634961,0.799353,0.707736,0.952028
8,0.035304,0.431197,0.793925,0.863187,0.825394,0.950546,0.631714,0.799353,0.705714,0.950546


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Sentiment Pairs

This section builds aspect-conditioned sentiment pairs, preserving the original review text and the exact aspect span for each annotation.

In [6]:
def build_sentiment_pairs(reviews: list[Review]) -> list[dict]:
    pairs = []
    seen = set()
    for review in reviews:
        for ann in review.annotations:
            key = (review.review_id, ann.start_char, ann.end_char, ann.sentiment.value)
            if key in seen:
                continue
            seen.add(key)
            pairs.append({
                'review_id': review.review_id,
                'product_id': review.product_id,
                'review_text': review.review_text,
                'aspect_text': ann.aspect,
                'start_char': ann.start_char,
                'end_char': ann.end_char,
                'label': ann.sentiment.value,
                'label_id': SENTIMENT_LABEL_TO_ID[ann.sentiment.value],
            })
    return pairs


def tokenize_sentiment_batch(batch: dict, tokenizer, max_length: int = MAX_LENGTH) -> dict:
    encoded = tokenizer(batch['review_text'], batch['aspect_text'], truncation='only_first', max_length=max_length, padding=False)
    encoded['labels'] = batch['label_id']
    return encoded


def create_sentiment_model(model_name: str = MODEL_NAME):
    config = AutoConfig.from_pretrained(model_name, num_labels=3, id2label=SENTIMENT_ID_TO_LABEL, label2id=SENTIMENT_LABEL_TO_ID)
    return AutoModelForSequenceClassification.from_pretrained(model_name, config=config)


def compute_sentiment_metrics(eval_prediction) -> dict[str, float]:
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_precision': float(precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)[0]),
        'macro_recall': float(precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)[1]),
        'macro_f1': float(f1_score(labels, predictions, average='macro', zero_division=0)),
    }


def train_sentiment_transformer():
    if not TRANSFORMER_AVAILABLE:
        raise RuntimeError(TRANSFORMER_IMPORT_ERROR or 'transformers unavailable')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    pair_splits = {split: build_sentiment_pairs(rows) for split, rows in processed_splits.items()}
    datasets = {split: Dataset.from_list(rows).map(lambda batch: tokenize_sentiment_batch(batch, tokenizer), batched=True, remove_columns=['review_id', 'product_id', 'review_text', 'aspect_text', 'start_char', 'end_char', 'label', 'label_id']) for split, rows in pair_splits.items()}
    output_dir = MODELS_DIR / 'sentiment_classifier' / 'distilbert'
    output_dir.mkdir(parents=True, exist_ok=True)
    model = create_sentiment_model()
    args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-5,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        weight_decay=0.01,
        warmup_steps=WARMUP_STEPS,
        optim='adamw_torch',
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        save_total_limit=2,
        fp16=bool(torch.cuda.is_available()),
        bf16=False,
        tf32=supports_tf32(),
        dataloader_num_workers=2,
        ddp_find_unused_parameters=False,
        seed=SEED,
        data_seed=SEED,
        report_to=[],
        remove_unused_columns=False,
    )

    class_weights = None
    labels = [row['label_id'] for row in pair_splits['train']]
    label_counts = Counter(labels)
    if len(label_counts) == 3:
        total = sum(label_counts.values())
        class_weights = torch.tensor([total / label_counts[i] for i in range(3)], dtype=torch.float)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels_tensor = inputs.pop('labels')
            outputs = model(**inputs)
            loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device) if class_weights is not None else None)
            loss = loss_fct(outputs.logits, labels_tensor)
            return (loss, outputs) if return_outputs else loss

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=datasets['train'],
        eval_dataset=datasets['validation'],
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_sentiment_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    train_result = trainer.train()
    trainer.save_model(str(output_dir / 'best'))
    tokenizer.save_pretrained(str(output_dir / 'best'))
    validation_metrics = trainer.evaluate(datasets['validation'])
    test_prediction = trainer.predict(datasets['test'])
    test_metrics = compute_sentiment_metrics((test_prediction.predictions, test_prediction.label_ids))
    test_predictions = np.argmax(test_prediction.predictions, axis=-1)
    test_true = test_prediction.label_ids
    per_class_recall = precision_recall_fscore_support(test_true, test_predictions, labels=[0, 1, 2], average=None, zero_division=0)[1]
    sentiment_test = {
        'accuracy': test_metrics['accuracy'],
        'macro_precision': test_metrics['macro_precision'],
        'macro_recall': test_metrics['macro_recall'],
        'macro_f1': test_metrics['macro_f1'],
        'per_class_recall': {SENTIMENT_ID_TO_LABEL[i]: float(per_class_recall[i]) for i in range(3)},
        'confusion_matrix': confusion_matrix(test_true, test_predictions, labels=[0, 1, 2]).tolist(),
    }
    manifest = {
        'manifest_version': '1.0',
        'task_id': 'A3_sentiment_classification',
        'mode': 'transformer',
        'run_id': f'sentiment_distilbert_seed{SEED}',
        'reason': 'transformer_training_available',
        'model': {'name': MODEL_NAME, 'revision': None, 'tokenizer_name': MODEL_NAME, 'tokenizer_revision': None},
        'seed': SEED,
        'config': {'epochs': NUM_EPOCHS, 'learning_rate': 2e-5, 'train_batch_size': TRAIN_BATCH_SIZE, 'eval_batch_size': EVAL_BATCH_SIZE, 'max_length': MAX_LENGTH},
        'dataset_hashes': {
            'train': sha256_text(json.dumps(pair_splits['train'], sort_keys=True)),
            'validation': sha256_text(json.dumps(pair_splits['validation'], sort_keys=True)),
            'test': sha256_text(json.dumps(pair_splits['test'], sort_keys=True)),
        },
        'checkpoint': {'path': str(output_dir / 'best'), 'sha256': sha256_directory(output_dir / 'best')},
        'metrics_paths': {
            'validation': str(EVIDENCE_DIR / 'A3' / 'sentiment_validation.json'),
            'test': str(EVIDENCE_DIR / 'A3' / 'sentiment_test.json'),
            'confusion_matrix': str(EVIDENCE_DIR / 'A3' / 'sentiment_confusion_matrix.json'),
        },
        'runtime': {'cuda_available': bool(torch.cuda.is_available()), 'gpu_count': int(torch.cuda.device_count())},
    }
    write_json(EVIDENCE_DIR / 'A3' / 'sentiment_training_manifest.json', manifest)
    write_json(EVIDENCE_DIR / 'A3' / 'sentiment_validation.json', validation_metrics)
    write_json(EVIDENCE_DIR / 'A3' / 'sentiment_test.json', sentiment_test)
    write_json(EVIDENCE_DIR / 'A3' / 'sentiment_confusion_matrix.json', {'labels': ['positive', 'neutral', 'negative'], 'matrix': sentiment_test['confusion_matrix']})
    return {'manifest': manifest, 'validation_metrics': validation_metrics, 'test_metrics': sentiment_test, 'trainer': trainer, 'tokenizer': tokenizer, 'pair_splits': pair_splits}


sentiment_training_result = train_sentiment_transformer()
a3_summary = {
    'macro_f1': sentiment_training_result['test_metrics']['macro_f1'],
    'macro_recall': sentiment_training_result['test_metrics']['macro_recall'],
    'per_class_recall': sentiment_training_result['test_metrics']['per_class_recall'],
    'baseline_macro_f1': sentiment_baseline_metrics['test']['macro_f1'],
    'passed': sentiment_training_result['test_metrics']['macro_f1'] >= 0.65 and all(v >= 0.50 for v in sentiment_training_result['test_metrics']['per_class_recall'].values()) and sentiment_training_result['test_metrics']['macro_f1'] > sentiment_baseline_metrics['test']['macro_f1'],
}


Map:   0%|          | 0/5107 [00:00<?, ? examples/s]

Map:   0%|          | 0/309 [00:00<?, ? examples/s]

Map:   0%|          | 0/1445 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Run Summary

This cell collects the run artifacts and makes the A2/A3 metrics easy to inspect in Kaggle without relying on print statements.

A2 means the aspect extractor should reach exact-span F1 of at least 0.70 and beat the rule baseline on the held-out test split.
A3 means the sentiment classifier should reach macro F1 of at least 0.65, keep every class recall at or above 0.50, and beat the TF-IDF baseline on the held-out test split.

In [7]:
summary = {
    'model_name': MODEL_NAME,
    'seed': SEED,
    'data_mode': 'linked_sources',
    'dataset_links': DATASET_LINKS,
    'source_files_ready': model_ready(),
    'processed_stats': processed_stats,
    'baseline': {
        'aspect_rule_test': rule_aspect_metrics['test'],
        'sentiment_test': sentiment_baseline_metrics['test'],
    },
    'A2': {
        'manifest': aspect_training_result['manifest'],
        'validation_metrics': aspect_training_result['validation_metrics'],
        'test_metrics': aspect_training_result['test_metrics'],
        'gate_claim': a2_summary,
    },
    'A3': {
        'manifest': sentiment_training_result['manifest'],
        'validation_metrics': sentiment_training_result['validation_metrics'],
        'test_metrics': sentiment_training_result['test_metrics'],
        'gate_claim': a3_summary,
    },
    'artifacts': {
        'processed_dir': str(PROCESSED_DIR),
        'aspect_checkpoint': str(aspect_training_result['manifest']['checkpoint']['path']),
        'sentiment_checkpoint': str(sentiment_training_result['manifest']['checkpoint']['path']),
    },
}
summary


{'model_name': 'distilbert-base-uncased',
 'seed': 42,
 'data_mode': 'linked_sources',
 'dataset_links': {'laptop_acos': 'https://github.com/NUSTM/ACOS/tree/main/data/Laptop-ACOS',
  'semeval_2014_laptop': 'https://aclanthology.org/S14-2004/'},
 'source_files_ready': True,
 'processed_stats': {'input_reviews': 7921,
  'accepted_reviews': 7878,
  'rejected_reviews': 0,
  'duplicate_count': 43,
  'split_counts': {'train': 5936, 'validation': 326, 'test': 1616},
  'source_counts': {'test': 1616, 'validation': 326, 'train': 5936},
  'sentiment_counts': {'negative': 2390, 'neutral': 870, 'positive': 3601},
  'aspect_counts': {'unit': 21,
   'acer 11': 1,
   'ssd drive': 3,
   'computer': 178,
   'chrome os': 16,
   'chrome': 10,
   'keyboard': 244,
   'touchscreen chromebook': 1,
   'touch pad': 21,
   'keys': 31,
   'device': 56,
   'track pad': 28,
   'os': 38,
   'hdmi': 6,
   'touchscreen': 32,
   'acer': 22,
   'sound': 31,
   'power adapter': 3,
   'wi - fi': 3,
   'product': 83,
   '